# ML for HEP

## Jet classification with PyTorch

We will now work through a compact but complete example of a deep-learning analysis in High-Energy Physics. The aim is not merely to run a finished neural network. We will prepare the data, construct the model, train it, inspect its failures, and ask whether the final result makes physical sense.

### The Task

We will use the **hls4ml Jet High-Level Features** dataset, stored as `hls4ml_HLF.arff` when available locally. Our goal is to classify jets from proton-proton collisions into five categories:

- `g`: gluon jets, represented by label `0`;
- `q`: quark jets, represented by label `1`;
- `w`: hadronically decaying $W$ bosons, represented by label `2`;
- `z`: hadronically decaying $Z$ bosons, represented by label `3`;
- `t`: hadronically decaying top quarks, represented by label `4`.

Each jet is represented by **16 high-level features** describing properties such as its shape, mass, energy distribution, and constituent multiplicity.

The full dataset contains many events. To keep this suitable for a short course, we begin with a smaller random sample. Once the full pipeline works, you can always scale it up.

## Imports and hardware check

First, we import the packages that will be needed throughout the notebook. We also ask PyTorch whether an accelerator is available:

- **CUDA** is used for supported NVIDIA GPUs;
- **MPS** is used for supported Apple Silicon GPUs;
- otherwise, the notebook falls back to the **CPU**.

The same code can therefore run on several types of machines.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import pandas as pd
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA (NVIDIA GPU) available: {torch.cuda.is_available()}")
print(f"MPS (Apple Silicon GPU) available: {torch.backends.mps.is_available()}") # NOTE: I am developing on a MacBook Pro with M1 chip, so I have access to MPS. But it is always good to confirm the hardware before committing to any analysis.

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

PyTorch version: 2.7.0
CUDA (NVIDIA GPU) available: False
MPS (Apple Silicon GPU) available: True
Using device: mps


### ❓ Exercise 

**Q1:** **What is `nn` vs `F`?** Why does PyTorch separate them? (Hint: Think about stateful object-oriented layers vs stateless mathematical functions).

**Q2:** **Why do we use `DataLoader`** instead of just feeding the entire dataset into the model at once? What memory and optimization benefits does it provide?

**Q3:** **What is CUDA or MPS?** Why is GPU training so much faster than CPU training for deep learning?


## Load and Prepare the Dataset

We now prepare the dataset step by step before giving it to the neural network.

1. **Load the ARFF file:**  
   ARFF stands for **Attribute-Relation File Format**. It is a text-based format commonly used in machine learning that stores both the description of the dataset—such as feature names and data types—and the actual data values.

2. **Decode the class labels:**  
   The labels may initially be stored as byte objects such as `b'q'` or `b'g'`. We decode them into ordinary Python strings such as `'q'` and `'g'`, which are easier to inspect and work with.

3. **Convert labels into numbers:**  
   A neural network cannot directly learn from text labels. We therefore map each jet class to an integer, for example,

   ```text
   q → 0
   g → 1
   w → 2
   z → 3
   t → 4
   ```

   These integers tell PyTorch which class each event belongs to.

4. **Take a random subset:**  
   The complete dataset is quite large. To make the first training exercise faster, we randomly select a smaller number of events. Because the sample is chosen randomly, it should still approximately represent the full dataset.

5. **Split the data:**  
   We divide the selected events into a **training set** and a **test set**. The model learns from the training set, while the test set is kept aside to check how well the trained model performs on events it has not seen before.

6. **Normalize the input features:**  
   Different jet observables can have very different numerical ranges. We use `StandardScaler` to transform each feature so that it has approximately zero mean and unit variance. This prevents features with large numerical values from dominating the training.

   Importantly, the scaler is fitted only on the training data and then applied to the test data. This avoids accidentally passing information from the test set into the training procedure.

7. **Convert everything into PyTorch tensors:**  
   Finally, we convert the NumPy arrays into PyTorch tensors. The input features are stored as floating-point tensors, while the class labels are stored as integer tensors. These are the formats expected by the neural network and the classification loss function.

After these steps, the data are ready to be grouped into batches and passed through the neural network.

In [2]:
import os

torch.manual_seed(42)
np.random.seed(42)

if os.path.exists('hls4ml_HLF.arff'):
    print("Loading hls4ml_HLF.arff (this might take ~5-10 seconds)...")
    data, meta = arff.loadarff('hls4ml_HLF.arff')
    df = pd.DataFrame(data)
    if 'class' in df.columns and hasattr(df['class'].iloc[0], 'decode'):
        df['class'] = df['class'].str.decode('utf-8')
else:
    print("Loading dataset via OpenML (hls4ml_lhc_jets_hlf)...")
    from sklearn.datasets import fetch_openml
    openml_data = fetch_openml('hls4ml_lhc_jets_hlf', version=1, as_frame=True)
    df = openml_data.frame
    if 'target' in df.columns and 'class' not in df.columns:
        df['class'] = df['target']

# Class mapping
class_names = ['g', 'q', 'w', 'z', 't']
class_mapping = {name: idx for idx, name in enumerate(class_names)}
df['label'] = df['class'].map(class_mapping)

# Extract features and labels
feature_cols = [col for col in df.columns if col not in ['class', 'label', 'target']]
X = df[feature_cols].values
y = df['label'].values

# --- WORKSHOP SUBSAMPLING ---
# The full dataset has 830,000 samples. Training on all of them on a CPU could take 20+ minutes.
# We will use a fast subset of 20,000 samples for the workshop. 
# You can increase this to 100,000+ or the full dataset later to boost your final accuracy!
n_samples = 20000
indices = np.random.choice(len(X), n_samples, replace=False)
X_subset = X[indices]
y_subset = y[indices]

# Split into Train (80%) and Test (20%)
train_X_raw, test_X_raw, train_y, test_y = train_test_split(
    X_subset, y_subset, test_size=0.2, random_state=42, stratify=y_subset
)

# --- FEATURE NORMALIZATION ---
# Neural networks perform best when input features are standardized (zero mean, unit variance).
scaler = StandardScaler()
train_X_scaled = scaler.fit_transform(train_X_raw)
test_X_scaled = scaler.transform(test_X_raw)

# Convert to PyTorch tensors
train_X = torch.tensor(train_X_scaled, dtype=torch.float32)
test_X = torch.tensor(test_X_scaled, dtype=torch.float32)
train_Y = torch.tensor(train_y, dtype=torch.long)
test_Y = torch.tensor(test_y, dtype=torch.long)

print("\n--- Dataset Summary ---")
print(f"Total features: {train_X.shape[1]}")
print(f"Training samples: {train_X.shape[0]}")
print(f"Test samples: {test_X.shape[0]}")
print(f"Class counts in subset:\n{pd.Series(y_subset).map({v:k for k,v in class_mapping.items()}).value_counts()}")

Loading dataset via OpenML (hls4ml_lhc_jets_hlf)...

--- Dataset Summary ---
Total features: 16
Training samples: 16000
Test samples: 4000
Class counts in subset:
w    4093
g    4068
z    4033
t    4017
q    3789
Name: count, dtype: int64


### ❓ Exercise 

**Q4:** **What happens if classes overlap significantly?** How does that affect the theoretical maximum accuracy our model can achieve?

**Q5:** **Why do we normalize features using `StandardScaler`?** What would happen to the weights in the first layer if one feature ranged from `0` to `1` and another ranged from `0` to `1,000,000`? (Hint: Think about gradients and learning rates).

**Q6:** **Examine the class distributions.** Are they balanced or unbalanced? How does class imbalance affect evaluation metrics (e.g., standard accuracy vs balanced accuracy)?


## Define the Model Architecture
Here we define our custom neural network class by inheriting from `nn.Module`. 
In PyTorch, we define our layers in `__init__` and the network's forward logic in `forward`.

In [3]:
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        # Layer 1: Linear projection from inputs to hidden features
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        # Non-linear activation
        self.relu = nn.ReLU()
        # Layer 2: Projection from hidden features to class logits
        self.layer2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.layer2(x)
        return x

# Instantiate model
model = SimpleClassifier(
    input_dim=16,       # 16 High-Level Features
    hidden_dim=32,      # Size of the hidden layer representation
    num_classes=5       # 5 jet classes
)

print(model)
print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

SimpleClassifier(
  (layer1): Linear(in_features=16, out_features=32, bias=True)
  (relu): ReLU()
  (layer2): Linear(in_features=32, out_features=5, bias=True)
)
Total trainable parameters: 709


### ❓ Exercise 

**Q7:** **Parameter Scaling**: Change `hidden_dim` from `32` to `8`, `64`, `128`, and `256`. Note down how the parameter count changes. Can you compute the formula for the number of parameters in this model? (Hint: don't forget biases!)

**Q8:** **Going Deeper (Coding Challenge)**: Modify the `SimpleClassifier` class (or write a new class `DeepClassifier` below) to add a second hidden layer. Your network flow should be:

- `Linear(input_dim -> hidden_dim)`
- `ReLU()`
- `Linear(hidden_dim -> hidden_dim)`
- `ReLU()`
- `Linear(hidden_dim -> num_classes)`

**Q9:** **Dropout Regularization (Coding Challenge)**: Insert a dropout layer (`nn.Dropout(p=0.3)`) after the activation function(s) to mitigate overfitting. What does dropout do during training, and how does it behave during evaluation (`model.eval()`)?


## DataLoader, Optimizer, and Loss
Next, we prepare our `DataLoader` for training, and choose our loss function and optimizer. 
These three components dictate how batches are loaded, how errors are quantified, and how weights are adjusted.

In [4]:
# Create DataLoader to feed data in mini-batches during training
train_loader = DataLoader(
    TensorDataset(train_X, train_Y),
    batch_size=32,      # Feed 32 samples at a time
    shuffle=True        # Shuffle every epoch to prevent ordering bias
)

# Loss function: Multi-class Cross Entropy
criterion = nn.CrossEntropyLoss()

# Optimizer: Adam optimizer with learning rate 0.01
learning_rate = 0.01
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

### ❓ Exercise 

**Q10:** **Batch Size Sweep**: What happens if you change `batch_size` to `8`, `16`, `64`, or `128`? How does batch size affect training speed (seconds per epoch) and the smoothness of the loss curve?

**Q11:** **Optimizer Experimentation**: Replace `Adam` with standard stochastic gradient descent (`torch.optim.SGD(model.parameters(), lr=0.01)`). Does standard SGD learn faster or slower than Adam? Why does Adam converge faster in complex settings?

**Q12:** **L2 Regularization**: Add `weight_decay=1e-4` to the `Adam` optimizer. Try values of `0`, `1e-5`, `1e-3`, and `1e-2`. How does L2 weight decay combat overfitting?


## The Training Loop
The heart of deep learning in PyTorch is the training loop. We iterate over our epochs and mini-batches, performing forward passes, gradient calculations, and optimizer steps.

In [5]:
epochs = 20

# Move model to the selected hardware device (GPU or CPU)
model = model.to(device)

train_losses = []
train_accs = []

for epoch in range(epochs):
    model.train() # Set model to training mode (enables Dropout/BatchNorm)
    
    total_loss = 0.0
    correct = 0
    total = 0
    
    for batch_X, batch_Y in train_loader:
        # Move mini-batch to active device
        batch_X, batch_Y = batch_X.to(device), batch_Y.to(device)
        
        # 1. Zero out previous gradients
        optimizer.zero_grad()
        
        # 2. Forward pass: compute predictions (logits)
        logits = model(batch_X)
        
        # 3. Compute loss
        loss = criterion(logits, batch_Y)
        
        # 4. Backward pass: compute gradients of loss w.r.t parameters
        loss.backward()
        
        # 5. Optimizer step: update weights
        optimizer.step()
        
        # Collect batch statistics
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == batch_Y).sum().item()
        total += len(batch_Y)
        
    # Calculate epoch metrics
    epoch_loss = total_loss / len(train_loader)
    epoch_acc = 100.0 * correct / total
    
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)
    
    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Loss: {epoch_loss:.4f} | Train Accuracy: {epoch_acc:.2f}%")

Epoch 01/20 | Loss: 0.9445 | Train Accuracy: 67.12%
Epoch 02/20 | Loss: 0.8524 | Train Accuracy: 71.81%
Epoch 03/20 | Loss: 0.8146 | Train Accuracy: 72.65%
Epoch 04/20 | Loss: 0.7971 | Train Accuracy: 73.03%
Epoch 05/20 | Loss: 0.7842 | Train Accuracy: 73.06%
Epoch 06/20 | Loss: 0.7752 | Train Accuracy: 73.38%
Epoch 07/20 | Loss: 0.7722 | Train Accuracy: 73.60%
Epoch 08/20 | Loss: 0.7620 | Train Accuracy: 73.67%
Epoch 09/20 | Loss: 0.7582 | Train Accuracy: 73.63%
Epoch 10/20 | Loss: 0.7518 | Train Accuracy: 73.96%
Epoch 11/20 | Loss: 0.7502 | Train Accuracy: 74.04%
Epoch 12/20 | Loss: 0.7506 | Train Accuracy: 73.87%
Epoch 13/20 | Loss: 0.7458 | Train Accuracy: 73.89%
Epoch 14/20 | Loss: 0.7423 | Train Accuracy: 74.16%
Epoch 15/20 | Loss: 0.7366 | Train Accuracy: 74.57%
Epoch 16/20 | Loss: 0.7380 | Train Accuracy: 74.36%
Epoch 17/20 | Loss: 0.7354 | Train Accuracy: 74.57%
Epoch 18/20 | Loss: 0.7304 | Train Accuracy: 74.61%
Epoch 19/20 | Loss: 0.7312 | Train Accuracy: 74.44%
Epoch 20/20 

### ❓ Exercise 

**Q13:** **Underfitting vs. Overfitting**: Run the training for `5`, `20`, and `100` epochs. At what point does the training loss stop decreasing significantly? Does the model begin to overfit if trained for too long?

**Q14:** **Early Stopping Coding Challenge**: Modify the training loop below (or implement a new version) to incorporate **Early Stopping**.

- *Instruction:* Evaluate validation loss at the end of each epoch. Keep track of the best validation loss. If the validation loss does not improve for 5 consecutive epochs, print a message and break the loop early.

## Evaluate the Model
We must test our model on unseen data, **Since thats the final goal anyway**. During evaluation, we put the model in `.eval()` mode and wrap our code in `with torch.no_grad():` to turn off gradient computation (saving memory and compute).

In [6]:
model.eval() # Set model to evaluation mode (disables Dropout/BatchNorm)

# Move test dataset to the active device
test_X = test_X.to(device)
test_Y = test_Y.to(device)

# Disable gradient computations
with torch.no_grad():
    logits = model(test_X)
    preds = logits.argmax(dim=1)
    acc = (preds == test_Y).float().mean()

print(f"Test Accuracy: {acc.item() * 100:.2f}%")

Test Accuracy: 74.12%


### ❓ Exercise

**Q15:** **Inspect Predictions (Coding challenge)**: Write a quick snippet to print the first 10 predictions alongside their true labels. Identify which predictions are correct and which are wrong.

**Q16:** **Locate Misclassifications (Coding challenge)**: Print the index and feature values of 3 samples that the model predicted incorrectly. What might have confused the model?

**Q17:** **Confusion Matrix (Coding challenge)**: Use `sklearn.metrics.confusion_matrix` and `matplotlib.pyplot` to compute and plot a Confusion Matrix.

Which particle classes are most frequently confused with each other? For example, quark `q` vs gluon `g`, or W boson `w` vs Z boson `z`. Why does this make physical sense?

## Logits vs Softmax Probabilities
Our model outputs raw values called **logits**. To convert them into interpretable probabilities that sum to 1, we apply the Softmax activation.

In [7]:
# Apply Softmax along the class dimension (dim=1)
probabilities = torch.softmax(logits, dim=1)

# Display the first 5 test samples
for i in range(5):
    print(f"Sample {i+1}:")
    print(f"  Logits:        {logits[i].cpu().numpy()}")
    print(f"  Probabilities: {probabilities[i].cpu().numpy()} (Sum: {probabilities[i].sum().item():.2f})")
    print(f"  Prediction:    {class_names[preds[i].item()]} (Class {preds[i].item()})")
    print(f"  True Label:    {class_names[test_Y[i].item()]} (Class {test_Y[i].item()})\n")

Sample 1:
  Logits:        [ 0.4411444   0.570843   -0.20781091  0.09380799 -2.0222065 ]
  Probabilities: [0.28962058 0.32972875 0.15135324 0.20463651 0.02466094] (Sum: 1.00)
  Prediction:    q (Class 1)
  True Label:    w (Class 2)

Sample 2:
  Logits:        [-0.7224077   0.25743017 -1.6749004   1.2461461  -1.5958525 ]
  Probabilities: [0.08600207 0.22911161 0.03317772 0.6158019  0.0359068 ] (Sum: 1.00)
  Prediction:    z (Class 3)
  True Label:    z (Class 3)

Sample 3:
  Logits:        [-3.5942411 -2.0412083 -1.8226286  3.8388171 -1.7660123]
  Probabilities: [5.8520585e-04 2.7655549e-03 3.4412029e-03 9.8956633e-01 3.6416519e-03] (Sum: 1.00)
  Prediction:    z (Class 3)
  True Label:    z (Class 3)

Sample 4:
  Logits:        [  1.5566454    0.44581485 -12.248738   -26.741423     5.833282  ]
  Probabilities: [1.3637509e-02 4.4906312e-03 1.3776323e-08 6.9990781e-15 9.8187178e-01] (Sum: 1.00)
  Prediction:    t (Class 4)
  True Label:    t (Class 4)

Sample 5:
  Logits:        [  0.28

### ❓ Exercise

**Q18:** Can you recognize something familiar from statistical mechanics in the cross-entropy loss?

**Q19:** **CrossEntropyLoss Detail**: Look back at Cell 4 where we defined our loss function as `nn.CrossEntropyLoss()`. Notice that our model's last layer in Cell 3 is a `nn.Linear` layer that directly outputs raw logits, not probabilities. What should be passed from the model to `nn.CrossEntropyLoss()`?

**Q20:** **Why does `nn.CrossEntropyLoss` NOT want us to add a Softmax layer at the end of our model?**

What two operations are combined inside `nn.CrossEntropyLoss`?


**Q21:** **What is the numerical stability benefit** of combining Softmax and Log operations together instead of computing them separately? (Hint: Think about floating-point exponentiation overflow and underflow).


## Save and Load the Model
Once you have trained your model, you'll want to save its weights so you can deploy it later. In PyTorch, we save the `state_dict()` which contains the model's weight and bias matrices.

In [8]:
# Save the trained weights to a file
torch.save(model.state_dict(), "classifier.pt")
print("Model weights saved to classifier.pt!")

# To load, we must first instantiate the architecture
loaded_model = SimpleClassifier(input_dim=16, hidden_dim=32, num_classes=5)

# Load the weights into the architecture
loaded_model.load_state_dict(torch.load("classifier.pt"))
loaded_model = loaded_model.to(device)
loaded_model.eval()

# Double check validation accuracy matches exactly
with torch.no_grad():
    loaded_logits = loaded_model(test_X)
    loaded_preds = loaded_logits.argmax(dim=1)
    loaded_acc = (loaded_preds == test_Y).float().mean()

print(f"Loaded Model Test Accuracy: {loaded_acc.item() * 100:.2f}%")

Model weights saved to classifier.pt!
Loaded Model Test Accuracy: 74.12%


### ❓ Exercise 

**Q22:** Now it is time to put everything you have learned to the test.

Your objective is to modify the code across the cells, or write custom code in the scratch cell below, to achieve the **highest possible test accuracy** on the jet-classification dataset.

Explore some combination of the following:

1. **Scale up the Data**: Increase `n_samples` from `20000` to `50000`, `150000`, or use the entire `830,000` dataset.
2. **Wider/Deeper Model**: Add more layers and hidden units, for example layers of size `128` and `64`.
3. **Combat Overfitting**: Introduce `nn.Dropout(p=0.2)` or `nn.Dropout(p=0.4)` and weight decay.
4. **Learning Rate Scheduling**: Start with a larger learning rate and reduce it as training progresses.
5. **Batch Size Tuning**: Try batch sizes such as `16`, `32`, `64`, `128`, and `256`.
6. **Optimizer Experimentation**: Compare Adam, AdamW, RMSprop, or SGD with momentum.

Can you surpass **80% accuracy**? Record your architecture, training strategy, and best test accuracy.

In [ ]:
# Write and run your custom challenge code here!
# Hint: Define a new model, configure a custom DataLoader & optimizer, train and evaluate.

